In [1]:
# Import necessary modules
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report


In [2]:
# Define parameters

def calc_monthly_income(annual_income):
  return annual_income / 12

def month_loan_term(years):
  return years * 12

def calc_PMT(PV, t):
  i = 0.075 / 12 # For simiplicity of this simulation, all loans have a set annual interest rate of 7.5%
  PMT = PV * ((i * (1 + i)**t) / ((1 + i)**t - 1))
  return PMT

def calc_credit_score_threshold(monthly_income, PMT):
  percent_of_monthly_income = PMT / monthly_income

  if percent_of_monthly_income < 0.1:
    return 650
  elif percent_of_monthly_income < 0.2:
    return 740
  elif percent_of_monthly_income < 0.4:
    return 800
  else:
    return 851

def calc_loan_approval(annual_income, credit_score, loan_principal, years):
  monthly_income = calc_monthly_income(annual_income)
  t = month_loan_term(years)
  PMT = calc_PMT(loan_principal, t)
  credit_score_threshold = calc_credit_score_threshold(monthly_income, PMT)

  if credit_score >= credit_score_threshold:
    return "APPROVED"
  else:
    return "DENIED"


In [3]:
# Load and preview data
df = pd.read_csv("ML Example (Loan Approval).csv")
print(df.head(10))

   income_lower  income_upper Annual Income  Credit Score Loan Principal   \
0         10000         25000    $13,542.00           301      $18,837.00   
1         10000         25000    $20,486.00           506      $43,362.00   
2         10000         25000    $14,718.00           725      $41,918.00   
3         10000         25000    $12,898.00           459      $17,762.00   
4         10000         25000    $19,152.00           542      $30,644.00   
5         10000         25000    $21,543.00           785      $20,268.00   
6         10000         25000    $24,894.00           740       $2,597.00   
7         10000         25000    $23,976.00           314      $10,073.00   
8         10000         25000    $10,649.00           497      $44,008.00   
9         10000         25000    $12,933.00           309      $32,103.00   

   Years  months  PMT_calc_1  PMT_calc_2  PMT_calc_3 Monthly Payment  \
0      1      12     0.00625    0.006735    0.077633       $1,634.25   
1      6

In [4]:
# This dataset was generated using numerous intermittent columns in excel.
# Remove those columns as they are out of scope for this project.
out_of_scope_columns = ['income_lower', 'income_upper', 'months', 'PMT_calc_1', 'PMT_calc_2', 'PMT_calc_3', 'percent_of_monthly_income', 'below_10%_monthly_income', 'below_20%_monthly_income', 'below_40%_monthly_income', 'credit_score_threshold']
df.drop(columns=out_of_scope_columns, inplace=True)

print(df.head(10))

  Annual Income  Credit Score Loan Principal   Years Monthly Payment  \
0    $13,542.00           301      $18,837.00      1       $1,634.25   
1    $20,486.00           506      $43,362.00      6         $749.73   
2    $14,718.00           725      $41,918.00      4       $1,013.53   
3    $12,898.00           459      $17,762.00     10         $210.84   
4    $19,152.00           542      $30,644.00      2       $1,378.97   
5    $21,543.00           785      $20,268.00      6         $350.44   
6    $24,894.00           740       $2,597.00      7          $39.83   
7    $23,976.00           314      $10,073.00      5         $201.84   
8    $10,649.00           497      $44,008.00     10         $522.38   
9    $12,933.00           309      $32,103.00      6         $555.06   

  Loan Approval  
0        Denied  
1        Denied  
2        Denied  
3        Denied  
4        Denied  
5      Approved  
6      Approved  
7        Denied  
8        Denied  
9        Denied  


In [5]:
# While the dataset was created to be randomized, to further ensure random selection of training, test, and or validation datasets, scramble the orders of the rows in the dataframe.
df = df.sample(frac=1).reset_index(drop=True)
print(df.head(10))

  Annual Income  Credit Score Loan Principal   Years Monthly Payment  \
0    $26,742.00           594       $5,440.00      2         $244.80   
1    $94,406.00           573       $3,886.00      4          $93.96   
2   $104,889.00           514      $21,503.00      2         $967.63   
3    $19,152.00           542      $30,644.00      2       $1,378.97   
4    $22,148.00           832      $12,497.00      6         $216.07   
5    $93,629.00           395       $6,033.00      4         $145.87   
6    $69,533.00           662      $29,189.00      2       $1,313.49   
7    $24,716.00           314       $4,208.00      6          $72.76   
8    $52,168.00           304      $43,763.00      9         $558.46   
9    $10,805.00           452      $46,307.00      3       $1,440.44   

  Loan Approval  
0        Denied  
1      Approved  
2        Denied  
3        Denied  
4      Approved  
5        Denied  
6        Denied  
7        Denied  
8        Denied  
9        Denied  


In [6]:
# Explore data
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Annual Income    200 non-null    object
 1   Credit Score     200 non-null    int64 
 2   Loan Principal   200 non-null    object
 3   Years            200 non-null    int64 
 4   Monthly Payment  200 non-null    object
 5   Loan Approval    200 non-null    object
dtypes: int64(2), object(4)
memory usage: 9.5+ KB
None


In [7]:
# Clean data

# There are several columns whose currency values are formated as strings ($1,234.56). These must be converted to floats for proper processing.
currency_cols = ['Annual Income', 'Loan Principal ', 'Monthly Payment']

for col in currency_cols:
    df[col] = df[col].replace(r'[$,]', '', regex=True).astype(float)

print(df.head())
print(df.info())

   Annual Income  Credit Score  Loan Principal   Years  Monthly Payment  \
0        26742.0           594           5440.0      2           244.80   
1        94406.0           573           3886.0      4            93.96   
2       104889.0           514          21503.0      2           967.63   
3        19152.0           542          30644.0      2          1378.97   
4        22148.0           832          12497.0      6           216.07   

  Loan Approval  
0        Denied  
1      Approved  
2        Denied  
3        Denied  
4      Approved  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Annual Income    200 non-null    float64
 1   Credit Score     200 non-null    int64  
 2   Loan Principal   200 non-null    float64
 3   Years            200 non-null    int64  
 4   Monthly Payment  200 non-null    float64
 5   Loan Appro

In [8]:
# Split the features and target to prevent data leakage. This prevents the model from seeing all the correct labels prematurely.
X = df.drop("Loan Approval", axis=1)
y = df["Loan Approval"]

In [9]:
# Split into training and testing datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42) # Didn't realize that this module includes ways to randomize and stratify the dataset, but left the randomizer in the beginning just for future reference.

In [10]:
numeric_features = X.select_dtypes(include=['int64','float64']).columns
categorical_features = X.select_dtypes(include=['object','category']).columns

In [11]:
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [12]:
log_reg = LogisticRegression(max_iter=1000)

In [13]:
clf = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', log_reg)
])

clf.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['Annual Income', 'Credit Score', 'Loan Principal ', 'Years',
       'Monthly Payment'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index([], dtype='object'))])),
                ('model', LogisticRegression(max_iter=1000))])

In [14]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

Accuracy: 0.95
AUC: 0.9846547314578005
              precision    recall  f1-score   support

    Approved       1.00      0.88      0.94        17
      Denied       0.92      1.00      0.96        23

    accuracy                           0.95        40
   macro avg       0.96      0.94      0.95        40
weighted avg       0.95      0.95      0.95        40



In [15]:
# Lets test the model by creating an entirely new loan application.

income = 80000.0
credit = 780
loan = 35180.59
years = 3

In [16]:
# How does the model predict for new loan applications?
new_applicant = pd.DataFrame([{
    'Annual Income': income,
    'Credit Score': credit,
    'Loan Principal ': loan,
    'Years': years,
    'Monthly Payment': calc_PMT(loan, month_loan_term(years))
}])

In [17]:
clf.predict(new_applicant)


array(['Approved'], dtype=object)

In [18]:
probs = clf.predict_proba(new_applicant)[0]

In [19]:
print(probs[0])

0.8881479254703147


In [20]:
clf.classes_

array(['Approved', 'Denied'], dtype=object)

In [21]:
# The calc_loan_approval function was designed with the same logic used to generate the original training data.
# Lets see how well the model predicted against the actual approval logic.

loan_status = calc_loan_approval(income, credit, loan, years)

In [22]:
print(f"The model predicts a {round(probs[0]*100,2)}% likelihood that the loan will be APPROVED.")
print(f"It predicts a {round(probs[1]*100,2)}% chance of denial.\n\n")

print(f"The original loan review logic would have {loan_status} this application.")

The model predicts a 88.81% likelihood that the loan will be APPROVED.
It predicts a 11.19% chance of denial.


The original loan review logic would have APPROVED this application.
